In [5]:
import numpy as np

In [ ]:
R = 8.314

In [29]:
def calculate_reactor_cost(volume: float, base_cost: float, scaling_factor: float) -> float:
    """
    Function to calculate reactor cost
    :param volume: reactor volume in m3
    :param base_cost: cost of 1 m3 reactor
    :param scaling_factor: economices of scale exponent
    :return: reactor cost
    """
    return base_cost * volume ** scaling_factor

In [28]:
def calculate_annual_profit(
    conversion: float,
    production_rate: float,
    product_price: float,
    raw_material_cost: float,
    energy_cost_per_year: float,
    labor_cost: float,
    maintenance_cost: float
) -> float:
    """
    Function to calculate annual profit
    :param conversion: Fractional conversion achieved
    :param production_rate: kg/year of product at 100% conversion
    :param product_price: $/kg selling price
    :param raw_material_cost: $/year for feedstock
    :param energy_cost_per_year: $/year for heating/cooling
    :param labor_cost: $/year labor cost
    :param maintenance_cost: $/year maintenance cost
    :return: Annual profit in dollars
    """
    return production_rate * conversion * product_price - (raw_material_cost + energy_cost_per_year + labor_cost + maintenance_cost)

In [ ]:
def calculate_npv(capital_cost: float, annual_profit: float, project_lifetime: int, discount_rate: float) -> float:
    """
    Function to calculate NPV
    :param capital_cost: Upfront investment in dollars
    :param annual_profit: Annual profit in dollars
    :param project_lifetime: Project duration in years
    :param discount_rate: In decimal (e.g., 0.10 for 10%)
    :return: Net Present Value in dollars
    """
    return np.sum(annual_profit / (1 + discount_rate) ** np.arange(1, project_lifetime + 1)) - capital_cost

In [ ]:
def optimize_reactor_design(param_ranges: dict, fixed_params: dict, optimization_target: str) -> dict:
    """
    Function to perform 2D optimization over temperature and time
    :param param_ranges: Ranges for temperature and time to test
    :param fixed_params: Kinetic parameters, economic parameters, etc.
    :param optimization_target: 'npv', 'conversion', or 'profit'
    :return: Dictionary with 'optimal_temp', 'optimal_time', 'optimal_value', 'optimization_surface'
    """
    temp_range = np.linspace(param_ranges['temperature'][0], param_ranges['temperature'][1], 100)
    time_range = np.linspace(param_ranges['time'][0], param_ranges['time'][1], 100)

    temp, time = np.meshgrid(temp_range, time_range)
    
    a = fixed_params['A']
    Ea = fixed_params['Ea']

    k = a * np.exp(-Ea / (R * temp))
    x = 1 - np.exp(-k * time)

    surface = x

    if optimization_target == 'conversion':
        pass
    elif optimization_target == 'profit':
        production_rate = fixed_params['production_rate']
        product_price = fixed_params['product_price']
        raw_material_cost = fixed_params['raw_material_cost']
        energy_cost = fixed_params['energy_cost']
        labor_cost = fixed_params['labor_cost']
        maintenance_cost = fixed_params['maintenance_cost']

        revenue = production_rate * x * product_price
        cost = raw_material_cost + energy_cost + labor_cost + maintenance_cost
        surface = revenue - cost
    else:
        capital_cost = fixed_params['capital_cost']
        project_lifetime = fixed_params['project_lifetime']
        discount_rate = fixed_params['discount_rate']

        production_rate = fixed_params['production_rate']
        product_price = fixed_params['product_price']
        raw_material_cost = fixed_params['raw_material_cost']
        energy_cost = fixed_params['energy_cost']
        labor_cost = fixed_params['labor_cost']
        maintenance_cost = fixed_params['maintenance_cost']

        revenue = production_rate * x * product_price
        annual_profit = revenue - (raw_material_cost + energy_cost + labor_cost + maintenance_cost)

        years = np.arange(1, project_lifetime + 1).reshape(-1, 1, 1)
        surface = np.sum(annual_profit / ((1 + discount_rate) ** years), axis=0) - capital_cost

    flat_index = np.argmax(surface)
    r, c = np.unravel_index(flat_index, surface.shape)
    optimal_temp = temp[r, c]
    optimal_time = time[r, c]
    optimal_value = surface[r, c]

    return {
        'optimal_temp': optimal_temp,
        'optimal_time': optimal_time,
        'optimal_value': optimal_value,
        'optimization_surface': surface
    }

In [ ]:
def sensitivity_analysis(base_case: dict, sensitivity_params: list, vary_range: float) -> dict:
    """
    Function to perform sensitivity analysis on NPV
    :param base_case: Base case design with all parameters
    :param sensitivity_params: Parameters to vary (e.g., ['product_price', 'energy_cost'])
    :param vary_range: Fraction to vary (e.g., 0.20 for ±20%)
    :return: Dictionary with parameter names as keys, values are dicts with 'low', 'base', 'high', sensitivity
    """

    output = {}

    for param in sensitivity_params:
        low_case = base_case.copy()
        high_case = base_case.copy()

        base_value = base_case[param]

        low_case[param] = base_value * (1 - vary_range)
        high_case[param] = base_value * (1 + vary_range)

        npv_base = calculate_npv(**base_case)
        npv_low = calculate_npv(**low_case)
        npv_high = calculate_npv(**high_case)

        sensitivity = (npv_high - npv_low) / (2 * vary_range * npv_base)

        output[param] = {
            'low': npv_low,
            'base': npv_base,
            'high': npv_high,
            'sensitivity': sensitivity
        }
    return output